In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully!")

In [ ]:
df = pd.read_csv("../data/ipl.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()


In [ ]:
df["match_id"].nunique()

In [ ]:
df["event_name"].value_counts()

In [ ]:
[c for c in df.columns if "season" in c.lower()]
df["season"].value_counts()

df["season"].dtype
df["season"].map(type).value_counts()
sorted(df["season"].astype(str).unique())

df["season"] = df["season"].astype(str)
df["season"].map(type).value_counts()
df["season"].value_counts().sort_index()

In [ ]:
df["batting_team"].unique()

In [ ]:
df["win_outcome"].value_counts()

In [ ]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
df["batting_team"].nunique()

In [ ]:
df["date"].min(), df["date"].max()

In [ ]:
df["date"] = pd.to_datetime(df["date"], dayfirst=True)

In [ ]:
df["date"].min(), df["date"].max()

In [ ]:
df["season"].value_counts().sort_index()

In [ ]:
matches = df.drop_duplicates("match_id").copy()
matches.shape

In [ ]:
df.groupby("season")["date"].agg(["min", "max"])

In [ ]:
# Check columns with unique values
df.nunique().sort_values()

In [ ]:
# Identify columns containing only one unique value
df.nunique()[df.nunique() == 1]

In [ ]:
# Check completely duplicated rows
df.duplicated().sum()

In [ ]:
df["method"].unique()

In [ ]:
df["power_surge_start"].unique()

In [ ]:
df["match_type"].unique()

In [ ]:
#cleaned dataset
df_clean = df.copy()
columns_to_remove = [
    "match_type",
    "event_name",
    "gender",
    "team_type",
    "balls_per_over",
    "overs",
    "match_number",
    "power_surge_start"
]

df_clean = df_clean.drop(columns=columns_to_remove)
df_clean.shape

In [ ]:
missing = df_clean.isnull().sum()

missing[missing > 0].sort_values(ascending=False)

In [ ]:
df_clean["win_outcome"].value_counts(dropna=False)

In [ ]:
df_clean["result_type"].value_counts(dropna=False)

In [ ]:
df_clean["method"].value_counts(dropna=False)

In [ ]:
df_clean["wicket_kind"].value_counts(dropna=False)

In [ ]:
match_check = df_clean.groupby("match_id").agg({
    "season": "nunique",
    "date": "nunique",
    "toss_winner": "nunique",
    "toss_decision": "nunique",
    "venue": "nunique",
    "match_won_by": "nunique",
    "result_type": "nunique"
})

match_check.describe()

In [ ]:
match_check[
    (match_check["season"] > 1) |
    (match_check["date"] > 1) |
    (match_check["toss_winner"] > 1) |
    (match_check["toss_decision"] > 1) |
    (match_check["venue"] > 1)
]

In [ ]:
matches = df_clean.groupby("match_id").agg({
    "season": "first",
    "date": "first",
    "toss_winner": "first",
    "toss_decision": "first",
    "venue": "first",
    "city": "first",
    "match_won_by": "first",
    "win_outcome": "first",
    "result_type": "first",
    "method": "first"
}).reset_index()

In [ ]:
matches.shape

In [ ]:
matches.head()

In [ ]:
teams_per_match = df_clean.groupby("match_id").agg({
    "batting_team": "nunique",
    "bowling_team": "nunique"
})

teams_per_match.head()

In [ ]:
#differentiating two teams
teams = (
    pd.concat([
        df_clean[["match_id", "batting_team"]]
        .rename(columns={"batting_team": "team"}),

        df_clean[["match_id", "bowling_team"]]
        .rename(columns={"bowling_team": "team"})
    ])
    .drop_duplicates()
)

In [ ]:
teams.head()

In [ ]:
teams = (
    teams.groupby("match_id")["team"]
    .apply(list)
    .reset_index()
)

In [ ]:
teams["team1"] = teams["team"].apply(lambda x: x[0])
teams["team2"] = teams["team"].apply(lambda x: x[1])

In [ ]:
teams = teams.drop(columns=["team"])

In [ ]:
teams.head()

In [ ]:
matches = df_clean.groupby("match_id").agg({
    "season": "first",
    "date": "first",
    "toss_winner": "first",
    "toss_decision": "first",
    "venue": "first",
    "city": "first",
    "match_won_by": "first",
    "win_outcome": "first",
    "result_type": "first",
    "method": "first"
}).reset_index()

In [ ]:
matches = matches.merge(
    teams,
    on="match_id",
    how="left"
)

In [ ]:
matches.shape

In [ ]:
matches.head()

In [ ]:
matches["match_won_by"].value_counts(dropna=False)

In [ ]:
matches["match_won_by"].isna().sum()

In [ ]:
matches["winner"] = matches["match_won_by"]

In [ ]:
matches[["match_id", "team1", "team2", "winner"]].head(10)

In [ ]:
matches["winner_valid"] = (
    (matches["winner"] == matches["team1"]) |
    (matches["winner"] == matches["team2"]) |
    (matches["winner"] == "Unknown")
)

In [ ]:
matches["winner_valid"].value_counts()

In [ ]:
matches[matches["winner"] == "Unknown"]

In [ ]:
#team wins analytical data
team_wins = (
    matches[matches["winner"] != "Unknown"]
    ["winner"]
    .value_counts()
    .reset_index()
)

In [ ]:
team_wins.columns = ["team", "wins"]
print(team_wins)

In [ ]:
#winner btn matches
team_matches = pd.concat([
    matches[["match_id", "team1", "winner"]]
        .rename(columns={"team1": "team"}),

    matches[["match_id", "team2", "winner"]]
        .rename(columns={"team2": "team"})
])

In [ ]:
team_matches.head()

In [ ]:
#matches played
matches_played = (
    team_matches
    .groupby("team")["match_id"]
    .nunique()
    .reset_index()
)

In [ ]:
matches_played.columns = ["team", "matches_played"]

In [ ]:
matches_played.sort_values(
    "matches_played",
    ascending=False
)

In [ ]:
#combining those two results
team_stats = matches_played.merge(
    team_wins,
    on="team",
    how="left"
)

In [ ]:
team_stats["wins"] = team_stats["wins"].fillna(0)

In [ ]:
team_stats["win_percentage"] = (
    team_stats["wins"] /
    team_stats["matches_played"] * 100
)

In [ ]:
team_stats = team_stats.sort_values(
    "win_percentage",
    ascending=False
)

In [ ]:
#displying
team_wins = (
    matches[matches["winner"] != "Unknown"]
    ["winner"]
    .value_counts()
    .reset_index()
)

team_wins.columns = ["team", "wins"]

In [ ]:
team_matches = pd.concat([
    matches[["match_id", "team1", "winner"]]
        .rename(columns={"team1": "team"}),

    matches[["match_id", "team2", "winner"]]
        .rename(columns={"team2": "team"})
])

In [ ]:
matches_played = (
    team_matches
    .groupby("team")["match_id"]
    .nunique()
    .reset_index()
)

matches_played.columns = ["team", "matches_played"]

In [ ]:
matches_played.sort_values("matches_played", ascending=False)

In [ ]:
#standardize team name
team_name_mapping = {
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Rising Pune Supergiant": "Rising Pune Supergiants"
}

In [ ]:
matches["team1"] = matches["team1"].replace(team_name_mapping)
matches["team2"] = matches["team2"].replace(team_name_mapping)
matches["winner"] = matches["winner"].replace(team_name_mapping)
matches["toss_winner"] = matches["toss_winner"].replace(team_name_mapping)

In [ ]:
#recreate team matches
team_matches = pd.concat([
    matches[["match_id", "team1", "winner"]]
        .rename(columns={"team1": "team"}),

    matches[["match_id", "team2", "winner"]]
        .rename(columns={"team2": "team"})
])

In [ ]:
matches_played = (
    team_matches
    .groupby("team")["match_id"]
    .nunique()
    .reset_index()
)

matches_played.columns = ["team", "matches_played"]

In [ ]:
team_wins = (
    matches[matches["winner"] != "Unknown"]
    ["winner"]
    .value_counts()
    .reset_index()
)

team_wins.columns = ["team", "wins"]

In [ ]:
team_stats = matches_played.merge(
    team_wins,
    on="team",
    how="left"
)

In [ ]:
team_stats["wins"] = team_stats["wins"].fillna(0)

In [ ]:
team_stats["win_percentage"] = (
    team_stats["wins"] /
    team_stats["matches_played"] * 100
)

In [ ]:
team_stats = team_stats.sort_values(
    "win_percentage",
    ascending=False
)

In [ ]:
print(team_stats)

In [ ]:
team_stats_50 = team_stats[
    team_stats["matches_played"] >= 50
].copy()

In [ ]:
team_stats_50 = team_stats_50.sort_values(
    "win_percentage",
    ascending=False
)

In [ ]:
print(team_stats_50)

In [ ]:
#chart of winning teams

plt.figure(figsize=(12, 7))

plt.bar(
    team_stats_50["team"],
    team_stats_50["win_percentage"]
)

plt.title("IPL Team Win Percentage (Minimum 50 Matches)")
plt.xlabel("Team")
plt.ylabel("Win Percentage (%)")

plt.xticks(rotation=60, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
#performance changed by seassons

season_matches = (
    matches.groupby("season")["match_id"]
    .nunique()
    .reset_index()
)

season_matches.columns = ["season", "matches"]

In [ ]:
print(season_matches)

In [ ]:
#series chart

plt.figure(figsize=(12, 6))

plt.plot(
    season_matches["season"].astype(str),
    season_matches["matches"],
    marker="o"
)

plt.title("Number of IPL Matches by Season")
plt.xlabel("Season")
plt.ylabel("Number of Matches")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
#runs per season

season_runs = (
    df_clean.groupby("season")["runs_total"]
    .sum()
    .reset_index()
)

In [ ]:
season_runs.columns = ["season", "total_runs"]

In [ ]:
print(season_runs)

In [ ]:
#combine matches and runs
season_analysis = season_matches.merge(
    season_runs,
    on="season",
    how="left"
)

In [ ]:
season_analysis["runs_per_match"] = (
    season_analysis["total_runs"] /
    season_analysis["matches"]
)

In [ ]:
print(season_analysis)

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    season_analysis["season"].astype(str),
    season_analysis["runs_per_match"],
    marker="o"
)

for i, value in enumerate(season_analysis["runs_per_match"]):
    plt.text(
        i,
        value + 3,
        f"{value:.0f}",
        ha="center",
        fontsize=8
    )

plt.title("Average Runs Scored per Match by IPL Season")
plt.xlabel("Season")
plt.ylabel("Runs per Match")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
#player performance analysis

batter_runs = (
    df_clean.groupby("batter")["runs_batter"]
    .sum()
    .reset_index()
)

batter_runs.columns = ["batter", "total_runs"]

batter_runs = batter_runs.sort_values(
    "total_runs",
    ascending=False
)

batter_runs.head(10)

In [ ]:
batter_balls = (
    df_clean.groupby("batter")["balls_faced"]
    .sum()
    .reset_index()
)

batter_balls.columns = ["batter", "balls_faced"]

In [ ]:
batter_stats = batter_runs.merge(
    batter_balls,
    on="batter",
    how="left"
)

In [ ]:
batter_stats.head(10)

In [ ]:
batter_stats["strike_rate"] = (
    batter_stats["total_runs"] /
    batter_stats["balls_faced"].replace(0, np.nan)
) * 100

In [ ]:
top_batters = batter_stats.sort_values(
    "total_runs",
    ascending=False
).head(10)

top_batters

In [ ]:
#chart for top batter

plt.figure(figsize=(12, 6))

plt.bar(
    top_batters["batter"],
    top_batters["total_runs"]
)

plt.title("Top 10 IPL Run Scorers")
plt.xlabel("Batter")
plt.ylabel("Total Runs")

plt.xticks(rotation=60, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
#bowling analysis

bowler_wicket_kinds = [
    "bowled",
    "caught",
    "lbw",
    "caught and bowled",
    "stumped",
    "hit wicket"
]

In [ ]:
bowler_wickets = df_clean[
    df_clean["wicket_kind"].isin(bowler_wicket_kinds)
]

In [ ]:
bowler_wickets.shape

In [ ]:
bowler_stats = (
    bowler_wickets
    .groupby("bowler")
    .size()
    .reset_index(name="wickets")
)

In [ ]:
bowler_stats = bowler_stats.sort_values(
    "wickets",
    ascending=False
)

In [ ]:
top_bowlers = bowler_stats.head(10)

top_bowlers

In [ ]:
#chart of top bowler analysis
plt.figure(figsize=(12, 6))

plt.bar(
    top_bowlers["bowler"],
    top_bowlers["wickets"]
)

plt.title("Top 10 IPL Wicket-Takers")
plt.xlabel("Bowler")
plt.ylabel("Wickets")

plt.xticks(rotation=60, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
#toss won and also match won analysis

matches["toss_match_winner"] = (
    matches["toss_winner"] == matches["winner"]
)

In [ ]:
toss_win_percentage = (
    matches["toss_match_winner"].mean() * 100
)

toss_win_percentage

In [ ]:
toss_analysis = (
    matches.groupby("toss_decision")["toss_match_winner"]
    .mean()
    .reset_index()
)

toss_analysis["win_percentage"] = (
    toss_analysis["toss_match_winner"] * 100
)

toss_analysis

In [ ]:
#chart of toss&win analysis

plt.figure(figsize=(8, 5))

plt.bar(
    toss_analysis["toss_decision"],
    toss_analysis["win_percentage"]
)

plt.title("Match Win Percentage by Toss Decision")
plt.xlabel("Toss Decision")
plt.ylabel("Toss Winner's Match Win Percentage (%)")

plt.tight_layout()
plt.show()

In [ ]:
matches["toss_decision"].unique()

In [ ]:
#venue analysis

venue_matches = (
    matches.groupby("venue")["match_id"]
    .nunique()
    .reset_index()
)

venue_matches.columns = ["venue", "matches"]

venue_matches = venue_matches.sort_values(
    "matches",
    ascending=False
)

venue_matches.head(10)

In [ ]:
venue_runs = (
    df_clean.groupby("venue")
    .agg(
        total_runs=("runs_total", "sum"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

In [ ]:
venue_runs["runs_per_match"] = (
    venue_runs["total_runs"] /
    venue_runs["matches"]
)

In [ ]:
venue_reliable = venue_runs[
    venue_runs["matches"] >= 20
].copy()

In [ ]:
venue_reliable = venue_reliable.sort_values(
    "runs_per_match",
    ascending=False
)

In [ ]:
venue_reliable.head(10)

In [ ]:
#chart for venue analysis
top_scoring_venues = venue_reliable.head(10)

plt.figure(figsize=(12, 6))

plt.barh(
    top_scoring_venues["venue"],
    top_scoring_venues["runs_per_match"]
)

plt.title("Highest Scoring IPL Venues")
plt.xlabel("Average Runs per Match")
plt.ylabel("Venue")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# FINAL DATASET FOR POWER BI
# ============================================

dashboard_columns = [
    "match_id",
    "date",
    "season",
    "innings",
    "batting_team",
    "bowling_team",
    "batter",
    "bowler",
    "runs_batter",
    "runs_total",
    "wicket_kind",
    "venue",
    "city",
    "toss_winner",
    "toss_decision",
    "match_won_by",
    "result_type",
    "method"
]

df_dashboard = df_clean[dashboard_columns].copy()

df_dashboard.head()

In [ ]:
df_dashboard.shape

In [ ]:
df_dashboard.info()

In [ ]:
#dashboard clean-up
# Standardize team names for dashboard

team_mapping = {
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Kings XI Punjab": "Punjab Kings",
    "Delhi Daredevils": "Delhi Capitals",
    "Deccan Chargers": "Deccan Chargers",
    "Rising Pune Supergiants": "Rising Pune Supergiant"
}

df_dashboard["batting_team_clean"] = (
    df_dashboard["batting_team"].replace(team_mapping)
)

df_dashboard["bowling_team_clean"] = (
    df_dashboard["bowling_team"].replace(team_mapping)
)

In [ ]:
df_dashboard["batting_team_clean"].value_counts()

In [ ]:
df_dashboard["batting_team_clean"].nunique()

In [ ]:
df_dashboard = df_dashboard.drop(
    columns=["batting_team", "bowling_team"]
)

df_dashboard = df_dashboard.rename(
    columns={
        "batting_team_clean": "batting_team",
        "bowling_team_clean": "bowling_team"
    }
)

In [ ]:
df_dashboard.shape

In [ ]:
df_dashboard.columns.tolist()

In [ ]:
# ============================================
# CREATE CLEAN DASHBOARD DATASET
# ============================================

df_clean = df.copy()

print("Original shape:", df_clean.shape)

In [ ]:
columns_to_remove = [
    "review_batter",
    "team_reviewed",
    "review_decision",
    "umpire",
    "player_of_match",
    "superover_winner",
    "event_match_no",
    "match_number",
    "new_batter",
    "power_surge_start",
    "batting_partners",
    "striker_out",
    "next_batter",
    "non_striker",
    "non_striker_pos",
    "fielders",
    "runs_not_boundary",
    "runs_bowler",
    "runs_not_boundary"
]

df_clean = df_clean.drop(
    columns=columns_to_remove,
    errors="ignore"
)

print("After removing unnecessary columns:", df_clean.shape)

In [ ]:
df_clean.shape

In [ ]:
df_clean.dtypes

In [ ]:
df_clean.isnull().sum().sort_values(ascending=False).head(15)

In [ ]:
df_clean["season"] = df_clean["season"].astype(str)

In [ ]:
df_clean["season"].dtype

In [ ]:
team_mapping = {
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Kings XI Punjab": "Punjab Kings",
    "Delhi Daredevils": "Delhi Capitals",
    "Rising Pune Supergiants": "Rising Pune Supergiant"
}

df_clean["batting_team"] = df_clean["batting_team"].replace(team_mapping)
df_clean["bowling_team"] = df_clean["bowling_team"].replace(team_mapping)

In [ ]:
df_clean["batting_team"].nunique()

In [ ]:
df_clean["batting_team"].unique()

In [ ]:
df_clean.duplicated().sum()

In [ ]:
numeric_columns = [
    "match_id",
    "innings",
    "over",
    "ball",
    "ball_no",
    "bat_pos",
    "runs_batter",
    "balls_faced",
    "valid_ball",
    "runs_extras",
    "runs_total",
    "team_runs",
    "team_balls",
    "team_wicket",
    "batter_runs",
    "batter_balls",
    "bowler_wicket"
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

In [ ]:
df_clean[numeric_columns].isnull().sum().sum()

In [ ]:
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))
print("Duplicate rows:", df_clean.duplicated().sum())
print("Teams:", df_clean["batting_team"].nunique())
print("Seasons:", df_clean["season"].nunique())

In [ ]:
#clean dataset uploaded
import os

os.makedirs("../data", exist_ok=True)

df_clean.to_csv(
    "../data/IPL_Dashboard_Data.csv",
    index=False
)

print("Clean dataset exported successfully!")

In [ ]:
os.path.exists("../data/IPL_Dashboard_Data.csv")

In [ ]:
df["season"] = df["season"].astype(str)

In [ ]:
seasons = df["season"].dropna().unique().tolist()